In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from numpy.polynomial.polynomial import Polynomial
import os

VIDEO_FILE = 'filepath'
SKIP_START = 10 #n frames to skip to front of video
SKIP_END = 5 #n frames to skip at end of video
SMOOTHING_WINDOW = 5 #window for rolling mean smoothing

def extract_variability(video_path, normalize='zscore', skip_start=SKIP_START, skip_end=SKIP_END):
    cap = cv2.VideoCapture(video_path)

    frame_idx = 0
    intensities = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        mean_intensity = np.mean(gray)
        intensities.append(mean_intensity)
        frame_idx += 1

    cap.release()
    print(f"Total frames read: {len(intensities)}")

    # Exclude first and last N frames
    total_frames = len(intensities)
    if total_frames <= (skip_start + skip_end):
        raise ValueError("Not enough frames in video after excluding first and last frames.")

    intensities = intensities[skip_start:total_frames - skip_end]
    frame_range = np.arange(skip_start, skip_start + len(intensities))
    print(f"Total frames read: {len(intensities)}")

    # Create DataFrame
    df = pd.DataFrame({
        'frame': frame_range,
        'mean_intensity': intensities
    })

    # Normalize
    if normalize == 'zscore':
        scaler = StandardScaler()
        df['variability'] = scaler.fit_transform(df[['mean_intensity']])
    elif normalize == 'minmax':
        scaler = MinMaxScaler()
        df['variability'] = scaler.fit_transform(df[['mean_intensity']])
    else:
        df['variability'] = df['mean_intensity']

    # Polynomial detrending (2nd order)
    x = df['frame'].values
    y = df['variability'].values
    poly_model = Polynomial.fit(x, y, 2).convert()
    trend = poly_model(x)
    df['trend'] = trend
    # Detrended signal (raw)
    df['detrended_raw'] = df['variability'] - trend

    # Apply rolling mean smoothing
    df['detrended'] = df['detrended_raw'].rolling(window=SMOOTHING_WINDOW, center=True, min_periods=1).mean()

    return df

# -------- Input and output configuration --------
video_file = VIDEO_FILE
video_id = os.path.splitext(os.path.basename(video_file))[0]
csv_filename = f'{video_id}.csv'
plot_filename = f'{video_id}_plot_original_and_detrended.png'

# -------- Run analysis --------
df = extract_variability(video_file, normalize='zscore', skip_start=SKIP_START, skip_end=SKIP_END)

# -------- Plot and save figure --------
plt.figure(figsize=(12, 6))

plt.subplot(2, 1, 1)
plt.plot(df['frame'], df['variability'], label='Normalized Intensity')
plt.plot(df['frame'], df['trend'], '--', label='2nd-Order Polynomial Trend')
plt.title('Signal with Trend')
plt.xlabel('Frame')
plt.ylabel('Z-Scored Intensity')
plt.legend()
plt.grid(True)

plt.subplot(2, 1, 2)
plt.plot(df['frame'], df['detrended_raw'], color='gray', alpha=0.4, label='Raw Detrended')
plt.plot(df['frame'], df['detrended'], color='purple', label='Smoothed Detrended')
plt.axhline(0, color='gray', linestyle='--', linewidth=0.5)
plt.title('Detrended Variability (Residuals)')
plt.xlabel('Frame')
plt.ylabel('Residual Intensity')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(plot_filename, dpi=300)
plt.show()

# -------- Save output to CSV --------
df.to_csv(csv_filename, index=False)

In [ ]:
from scipy.stats import skew, kurtosis, linregress
from scipy.signal import periodogram, find_peaks
from statsmodels.tsa.stattools import acf
import antropy as ant
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ---------- Extract smoothed residual series ----------
residual = df['detrended'].values  # smoothed version
time = df['frame'].values
n = len(residual)

# ---------- Trend Features ----------
poly3 = np.polyfit(time, residual, 3)
trend_coef_1 = poly3[1]
trend_coef_2 = poly3[2]
trend_coef_3 = poly3[3]

# ---------- Cyclic/Oscillatory Features ----------
freqs, powers = periodogram(residual)
dominant_freq = freqs[np.argmax(powers)] if len(freqs) > 0 else 0
spectral_entropy = ant.spectral_entropy(residual, sf=1.0, normalize=True)

# ---------- Variance Features ----------
rolling_std = pd.Series(residual).rolling(window=10, center=True).std()
rolling_std_range = rolling_std.max() - rolling_std.min()

# ---------- Noise/Randomness Features ----------
try:
    sample_entropy = ant.sample_entropy(residual)
except:
    sample_entropy = np.nan

autocorr_vals = acf(residual, nlags=2, fft=False)
lag1_autocorr = autocorr_vals[1] if len(autocorr_vals) > 1 else np.nan

# ---------- Summary Statistics ----------
residual_mean = np.mean(residual)
residual_median = np.median(residual)
residual_min = np.min(residual)
residual_max = np.max(residual)
residual_std = np.std(residual)

# ---------- Combine Features ----------
features = {
    'video_id': video_id,
    'trend_coef_1_linear': trend_coef_1,
    'trend_coef_2_quadratic': trend_coef_2,
    'trend_coef_3_cubic': trend_coef_3,
    'dominant_freq': dominant_freq,
    'spectral_entropy': spectral_entropy,
    'rolling_std_range': rolling_std_range,
    'sample_entropy': sample_entropy,
    'lag1_autocorrelation': lag1_autocorr,
    'residual_mean': residual_mean,
    'residual_median': residual_median,
    'residual_min': residual_min,
    'residual_max': residual_max,
    'residual_std': residual_std
}

features_df = pd.DataFrame([features])
features_filename = f'{video_id}_features.csv'
features_df.to_csv(features_filename, index=False)

# ---------- Optional: Preview Feature Table in Notebook ----------
features_df.T